# 🚀 AccessiCap Cloud Training

Train your AI model using the code from your GitHub repository.

**Repository:** [https://github.com/zoenut/AccessiCap](https://github.com/zoenut/AccessiCap)

### **Steps:**
1. **Runtime → Change runtime type → T4 GPU**
2. Upload `dataset.zip` to your **Google Drive (My Drive root)** once — you never need to re-upload.
3. Run all cells in order (`Runtime → Run all`).
4. The trained model will download automatically when training finishes.

In [ ]:
# @title 1. Prepare Environment & Clone Repo
!pip install -q transformers==4.46.0 torch torchvision pillow accelerate peft evaluate datasets

# Clone the repository (skip if already cloned)
import os
if not os.path.exists('AccessiCap'):
    !git clone https://github.com/zoenut/AccessiCap.git
else:
    print('✅ Repo already cloned — pulling latest changes...')
    !git -C AccessiCap pull

%cd AccessiCap/backend/training
print('✅ Environment ready!')

In [ ]:
# @title 2. Load Dataset from Google Drive
# Upload dataset.zip to the ROOT of your Google Drive once.
# This cell will mount Drive and copy it automatically every session.

import os, shutil, zipfile
from google.colab import drive

DRIVE_DATASET_PATH = '/content/drive/MyDrive/dataset.zip'   # ← change if you store it elsewhere
LOCAL_ZIP          = '/content/AccessiCap/backend/training/dataset.zip'
DATASET_DIR        = '/content/AccessiCap/backend/training/dataset'

if os.path.exists(DATASET_DIR):
    print('✅ Dataset already extracted — skipping.')
else:
    # Mount Google Drive
    print('📂 Mounting Google Drive...')
    drive.mount('/content/drive')

    if os.path.exists(DRIVE_DATASET_PATH):
        print(f'📋 Copying dataset.zip from Drive (~1 GB, please wait)...')
        shutil.copy(DRIVE_DATASET_PATH, LOCAL_ZIP)
        print('📦 Extracting dataset...')
        with zipfile.ZipFile(LOCAL_ZIP, 'r') as z:
            z.extractall(DATASET_DIR)
        os.remove(LOCAL_ZIP)   # free space
        print('✅ Dataset ready!')
    else:
        print('❌ dataset.zip not found in My Drive.')
        print(f'   Expected path: {DRIVE_DATASET_PATH}')
        print('   Please upload dataset.zip to the root of your Google Drive and re-run this cell.')

In [ ]:
# @title 3. Start Training ⚡  (≈ 30–60 min on T4 GPU)
# Uses LoRA fine-tuning — only 1-3% of model params are trained, very efficient.
# batch_size=4 is safe for T4 (16 GB VRAM); increase to 8 if you have an A100.

!python train_blip.py \
    --dataset_path ./dataset \
    --use_lora \
    --epochs 10 \
    --batch_size 4 \
    --learning_rate 2e-5 \
    --output_dir ./final_model

In [ ]:
# @title 4. Save Model Back to Google Drive (optional but recommended)
# This saves the trained model to your Drive so you don't lose it when the session ends.

import shutil, os
from google.colab import drive

# Make sure Drive is mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

DRIVE_SAVE_PATH = '/content/drive/MyDrive/accessicap_model'

if os.path.exists('./final_model'):
    print('💾 Copying model to Google Drive...')
    shutil.copytree('./final_model', DRIVE_SAVE_PATH, dirs_exist_ok=True)
    print(f'✅ Model saved to Google Drive at: {DRIVE_SAVE_PATH}')
else:
    print('❌ ./final_model not found — did training complete successfully?')

In [ ]:
# @title 5. Download Trained Model to your Computer
from google.colab import files
import os

if os.path.exists('./final_model'):
    print('📦 Zipping model...')
    !zip -r accessicap_model.zip ./final_model
    print('⬇️  Downloading accessicap_model.zip ...')
    files.download('accessicap_model.zip')
else:
    print('❌ ./final_model not found — did training complete successfully?')